In [1]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-GN-8B-GN-PN-20-800ed9dc",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since protein names are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"protein_name": "Error"})

# Improved function to extract JSON from the LLM response for protein data
def extract_protein_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'protein_name' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"protein_name"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract protein_name value directly
        protein_name_match = re.search(r'"protein_name"[\s:]*"([^"]+)"', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        protein_name_match = re.search(r'protein_name["\':\s]+([^"\'}\s,]+)', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"protein_name": "None"}

        # If all parsing attempts fail
        return {"protein_name": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"protein_name": "Parse_Error"}

# Enhanced prompt to get protein name for a given gene symbol
def get_protein_name_from_gene(gene_symbol):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the gene symbol "{gene_symbol}", provide the corresponding protein name.

Instructions:
- Provide the EXACT official protein name as commonly used in scientific literature
- Choose the most concise, standard form of the protein name
- Avoid overly technical or lengthy variations (e.g., prefer "p53" over "Cellular tumor antigen p53")
- Return the primary, commonly used protein name
- Use standard nomenclature when available
- Examples: 
  - "TP53" → "p53"
  - "BRCA1" → "Breast cancer type 1 susceptibility protein"
  - "INS" → "Insulin"
- If the gene symbol is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "protein_name": "<protein_name_here>"
}}

Gene Symbol: {gene_symbol}"""

    response_text = query_together(prompt)
    result = extract_protein_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "gene_symbol": gene_symbol,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("gene_to_protein_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("protein_name", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_protein_name, new_protein_name):
    """
    Calculate match result between original and new protein names
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_protein_name) or pd.isna(new_protein_name):
        return 0
    return int(str(original_protein_name).strip().lower() == str(new_protein_name).strip().lower())

# Main processing function
def main():
    print("Starting gene symbol to protein name mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("gene_to_protein_api_responses_finetuned.jsonl"):
        with open("gene_to_protein_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample gene symbol
    print("Testing API connection...")
    test_result = get_protein_name_from_gene("TP53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_protein_name"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            gene_symbol = row["GN"]  # Using the GN column from the CSV
            original_protein_name = row["protein_name"]  # Original protein name for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {gene_symbol}")

            start_time = time.time()
            new_protein_name = get_protein_name_from_gene(gene_symbol)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new protein name
            df.loc[idx, "new_protein_name"] = new_protein_name
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_protein_name, new_protein_name)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_protein_name}, New: {new_protein_name}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_gene_to_protein_results_progress20.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} gene symbols.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_protein_name"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_gene_to_protein_results_final20.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Gene to Protein) ===")
        print(f"Total gene symbols processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_protein_name"] == "None").sum()
        error_results = (df["new_protein_name"] == "Error").sum()
        parse_error_results = (df["new_protein_name"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid protein names returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_protein_name"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_gene_to_protein_results_final5.csv")
    print(f"Progress file: finetuned_gene_to_protein_results_progress5.csv")
    print(f"Log file: gene_to_protein_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting gene symbol to protein name mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: Tumor protein p53
API test successful, proceeding with batch processing...
Loaded 3978 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv
Columns in the dataset: ['ID', 'AC', 'GN', 'protein_name', 'GO_F Count', 'GO_P Count', 'GO_C Count', 'go_counts', 'bin', 'normalized_protein', 'normalized_gene_name', 'Match_GN', 'pmc_GN', 'PMC_protein_name']
Processing 1/3978: EXD2
Sending request (attempt 1/5)...
  Original: Exonuclease 3'-5' domain-containing protein 2 , New: Exonuclease 3'-5' domain-containing protein 2, Match: 1
Progress saved. Processed 1/3978 gene symbols.
Waiting 2.25 seconds before next request...
Processing 2/3978: EXOSC6
Sending request (attempt 1/5)...
  Original: Exosome complex component MTR3, New: Exosome complex component EXOSC6, Match: 0
Wait

15 eps

In [ ]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-Protein-8B-Gene-Protein-1-70326a78",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since protein names are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"protein_name": "Error"})

# Improved function to extract JSON from the LLM response for protein data
def extract_protein_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'protein_name' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"protein_name"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract protein_name value directly
        protein_name_match = re.search(r'"protein_name"[\s:]*"([^"]+)"', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        protein_name_match = re.search(r'protein_name["\':\s]+([^"\'}\s,]+)', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"protein_name": "None"}

        # If all parsing attempts fail
        return {"protein_name": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"protein_name": "Parse_Error"}

# Enhanced prompt to get protein name for a given gene symbol
def get_protein_name_from_gene(gene_symbol):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the gene symbol "{gene_symbol}", provide the corresponding protein name.

Instructions:
- Provide the EXACT official protein name as commonly used in scientific literature
- Choose the most concise, standard form of the protein name
- Avoid overly technical or lengthy variations (e.g., prefer "p53" over "Cellular tumor antigen p53")
- Return the primary, commonly used protein name
- Use standard nomenclature when available
- Examples: 
  - "TP53" → "p53"
  - "BRCA1" → "Breast cancer type 1 susceptibility protein"
  - "INS" → "Insulin"
- If the gene symbol is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "protein_name": "<protein_name_here>"
}}

Gene Symbol: {gene_symbol}"""

    response_text = query_together(prompt)
    result = extract_protein_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "gene_symbol": gene_symbol,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("gene_to_protein_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("protein_name", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_protein_name, new_protein_name):
    """
    Calculate match result between original and new protein names
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_protein_name) or pd.isna(new_protein_name):
        return 0
    return int(str(original_protein_name).strip().lower() == str(new_protein_name).strip().lower())

# Main processing function
def main():
    print("Starting gene symbol to protein name mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("gene_to_protein_api_responses_finetuned.jsonl"):
        with open("gene_to_protein_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample gene symbol
    print("Testing API connection...")
    test_result = get_protein_name_from_gene("TP53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_protein_name"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            gene_symbol = row["GN"]  # Using the GN column from the CSV
            original_protein_name = row["protein_name"]  # Original protein name for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {gene_symbol}")

            start_time = time.time()
            new_protein_name = get_protein_name_from_gene(gene_symbol)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new protein name
            df.loc[idx, "new_protein_name"] = new_protein_name
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_protein_name, new_protein_name)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_protein_name}, New: {new_protein_name}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_gene_to_protein_results_progress15.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} gene symbols.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_protein_name"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_gene_to_protein_results_final15.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Gene to Protein) ===")
        print(f"Total gene symbols processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_protein_name"] == "None").sum()
        error_results = (df["new_protein_name"] == "Error").sum()
        parse_error_results = (df["new_protein_name"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid protein names returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_protein_name"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_gene_to_protein_results_final5.csv")
    print(f"Progress file: finetuned_gene_to_protein_results_progress5.csv")
    print(f"Log file: gene_to_protein_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

10 eps

In [ ]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-Protein-8B-Gene-Protein-1-70326a78",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since protein names are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"protein_name": "Error"})

# Improved function to extract JSON from the LLM response for protein data
def extract_protein_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'protein_name' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"protein_name"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract protein_name value directly
        protein_name_match = re.search(r'"protein_name"[\s:]*"([^"]+)"', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        protein_name_match = re.search(r'protein_name["\':\s]+([^"\'}\s,]+)', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"protein_name": "None"}

        # If all parsing attempts fail
        return {"protein_name": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"protein_name": "Parse_Error"}

# Enhanced prompt to get protein name for a given gene symbol
def get_protein_name_from_gene(gene_symbol):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the gene symbol "{gene_symbol}", provide the corresponding protein name.

Instructions:
- Provide the EXACT official protein name as commonly used in scientific literature
- Choose the most concise, standard form of the protein name
- Avoid overly technical or lengthy variations (e.g., prefer "p53" over "Cellular tumor antigen p53")
- Return the primary, commonly used protein name
- Use standard nomenclature when available
- Examples: 
  - "TP53" → "p53"
  - "BRCA1" → "Breast cancer type 1 susceptibility protein"
  - "INS" → "Insulin"
- If the gene symbol is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "protein_name": "<protein_name_here>"
}}

Gene Symbol: {gene_symbol}"""

    response_text = query_together(prompt)
    result = extract_protein_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "gene_symbol": gene_symbol,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("gene_to_protein_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("protein_name", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_protein_name, new_protein_name):
    """
    Calculate match result between original and new protein names
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_protein_name) or pd.isna(new_protein_name):
        return 0
    return int(str(original_protein_name).strip().lower() == str(new_protein_name).strip().lower())

# Main processing function
def main():
    print("Starting gene symbol to protein name mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("gene_to_protein_api_responses_finetuned.jsonl"):
        with open("gene_to_protein_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample gene symbol
    print("Testing API connection...")
    test_result = get_protein_name_from_gene("TP53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_protein_name"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            gene_symbol = row["GN"]  # Using the GN column from the CSV
            original_protein_name = row["protein_name"]  # Original protein name for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {gene_symbol}")

            start_time = time.time()
            new_protein_name = get_protein_name_from_gene(gene_symbol)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new protein name
            df.loc[idx, "new_protein_name"] = new_protein_name
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_protein_name, new_protein_name)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_protein_name}, New: {new_protein_name}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_gene_to_protein_results_progress10.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} gene symbols.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_protein_name"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_gene_to_protein_results_final10.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Gene to Protein) ===")
        print(f"Total gene symbols processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_protein_name"] == "None").sum()
        error_results = (df["new_protein_name"] == "Error").sum()
        parse_error_results = (df["new_protein_name"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid protein names returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_protein_name"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_gene_to_protein_results_final5.csv")
    print(f"Progress file: finetuned_gene_to_protein_results_progress5.csv")
    print(f"Log file: gene_to_protein_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

5 eps

In [2]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-GN-8B-GN-PN-5-8e0552fd",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since protein names are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"protein_name": "Error"})

# Improved function to extract JSON from the LLM response for protein data
def extract_protein_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'protein_name' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"protein_name"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract protein_name value directly
        protein_name_match = re.search(r'"protein_name"[\s:]*"([^"]+)"', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        protein_name_match = re.search(r'protein_name["\':\s]+([^"\'}\s,]+)', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"protein_name": "None"}

        # If all parsing attempts fail
        return {"protein_name": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"protein_name": "Parse_Error"}

# Enhanced prompt to get protein name for a given gene symbol
def get_protein_name_from_gene(gene_symbol):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the gene symbol "{gene_symbol}", provide the corresponding protein name.

Instructions:
- Provide the EXACT official protein name as commonly used in scientific literature
- Choose the most concise, standard form of the protein name
- Avoid overly technical or lengthy variations (e.g., prefer "p53" over "Cellular tumor antigen p53")
- Return the primary, commonly used protein name
- Use standard nomenclature when available
- Examples: 
  - "TP53" → "p53"
  - "BRCA1" → "Breast cancer type 1 susceptibility protein"
  - "INS" → "Insulin"
- If the gene symbol is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "protein_name": "<protein_name_here>"
}}

Gene Symbol: {gene_symbol}"""

    response_text = query_together(prompt)
    result = extract_protein_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "gene_symbol": gene_symbol,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("gene_to_protein_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("protein_name", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_protein_name, new_protein_name):
    """
    Calculate match result between original and new protein names
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_protein_name) or pd.isna(new_protein_name):
        return 0
    return int(str(original_protein_name).strip().lower() == str(new_protein_name).strip().lower())

# Main processing function
def main():
    print("Starting gene symbol to protein name mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("gene_to_protein_api_responses_finetuned.jsonl"):
        with open("gene_to_protein_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample gene symbol
    print("Testing API connection...")
    test_result = get_protein_name_from_gene("TP53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_protein_name"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            gene_symbol = row["GN"]  # Using the GN column from the CSV
            original_protein_name = row["protein_name"]  # Original protein name for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {gene_symbol}")

            start_time = time.time()
            new_protein_name = get_protein_name_from_gene(gene_symbol)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new protein name
            df.loc[idx, "new_protein_name"] = new_protein_name
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_protein_name, new_protein_name)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_protein_name}, New: {new_protein_name}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_gene_to_protein_results_progress5.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} gene symbols.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_protein_name"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_gene_to_protein_results_final5.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Gene to Protein) ===")
        print(f"Total gene symbols processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_protein_name"] == "None").sum()
        error_results = (df["new_protein_name"] == "Error").sum()
        parse_error_results = (df["new_protein_name"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid protein names returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_protein_name"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_gene_to_protein_results_final5.csv")
    print(f"Progress file: finetuned_gene_to_protein_results_progress5.csv")
    print(f"Log file: gene_to_protein_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting gene symbol to protein name mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: Tumor protein p53
API test successful, proceeding with batch processing...
Loaded 3978 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv
Columns in the dataset: ['ID', 'AC', 'GN', 'protein_name', 'GO_F Count', 'GO_P Count', 'GO_C Count', 'go_counts', 'bin', 'normalized_protein', 'normalized_gene_name', 'Match_GN', 'pmc_GN', 'PMC_protein_name']
Processing 1/3978: EXD2
Sending request (attempt 1/5)...
  Original: Exonuclease 3'-5' domain-containing protein 2 , New: Exonuclease 3'-5' domain-containing protein 2, Match: 1
Progress saved. Processed 1/3978 gene symbols.
Waiting 2.76 seconds before next request...
Processing 2/3978: EXOSC6
Sending request (attempt 1/5)...
  Original: Exosome complex component MTR3, New: Exosome complex component EXOSC6, Match: 0
Wait

1 eps

In [3]:
import pandas as pd
import json
import re
import time
import random
import os
from datetime import datetime
from together import Together

# Define the Together API key - replace with your actual key
TOGETHER_API_KEY = "tgp_v1_NCCZx5luCkWXyhlO3KyxvBMgoxIG9i9V23UC8rI1s84"
# Initialize the Together client with the API key
client = Together(api_key=TOGETHER_API_KEY)

# Function to query Together LLM API with improved error handling and exponential backoff
def query_together(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            # Calculate exponential backoff with jitter
            if attempt > 0:
                base_delay = min(30, 2 ** attempt)  # Cap at 30 seconds
                jitter = random.uniform(0, 0.5 * base_delay)  # Add up to 50% jitter
                wait_time = base_delay + jitter
                print(f"Attempt {attempt+1}/{max_retries} - Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)

            print(f"Sending request (attempt {attempt+1}/{max_retries})...")
            # Send the request to the Together API with your fine-tuned model
            response = client.chat.completions.create(
                model="suswitha/Meta-Llama-3.1-8B-Instruct-Reference-GN-8B-GN-PN-1-426db02a",  # Your fine-tuned protein model
                messages=[
                    {"role": "system", "content": "You are a biomedical expert specializing in gene nomenclature and protein names. Always respond in valid JSON format with only the requested information."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.05,  # Lower temperature for more deterministic results from fine-tuned model
                max_tokens=128,  # Reduced since protein names are usually short
                top_p=0.9,  # Add top_p for better control over token selection
                response_format={"type": "json_object"}
            )
            
            # Get the content from the response
            content = response.choices[0].message.content
            return content

        except Exception as e:
            print(f"Request error: {str(e)}")
            
    # If all retries failed
    return json.dumps({"protein_name": "Error"})

# Improved function to extract JSON from the LLM response for protein data
def extract_protein_data(response_text):
    try:
        # First attempt: try to parse the entire response as JSON
        try:
            json_obj = json.loads(response_text.strip())
            if 'protein_name' in json_obj:
                return json_obj
        except json.JSONDecodeError:
            pass  # Continue to other extraction methods

        # Try to find JSON block in markdown code block
        json_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Try to find any JSON object in the text
        json_pattern = re.compile(r'(\{[^{]*?"protein_name"[^}]*?\})', re.DOTALL)
        json_match = json_pattern.search(response_text)
        if json_match:
            try:
                json_obj = json.loads(json_match.group(1))
                if 'protein_name' in json_obj:
                    return json_obj
            except json.JSONDecodeError:
                pass

        # Extract protein_name value directly
        protein_name_match = re.search(r'"protein_name"[\s:]*"([^"]+)"', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # If we still don't have valid JSON, try to extract fields directly
        protein_name_match = re.search(r'protein_name["\':\s]+([^"\'}\s,]+)', response_text)
        if protein_name_match:
            return {"protein_name": protein_name_match.group(1).strip()}

        # Final fallback: If the response seems to indicate no match
        if re.search(r'[Nn]o(ne| relevant| matching)', response_text) or "not found" in response_text.lower():
            return {"protein_name": "None"}

        # If all parsing attempts fail
        return {"protein_name": "Parse_Error"}

    except Exception as e:
        print(f"JSON parsing error: {e}")
        print(f"Response text: {response_text[:200]}...")
        return {"protein_name": "Parse_Error"}

# Enhanced prompt to get protein name for a given gene symbol
def get_protein_name_from_gene(gene_symbol):
    # Detailed prompt with instructions for your fine-tuned model
    prompt = f"""Given the gene symbol "{gene_symbol}", provide the corresponding protein name.

Instructions:
- Provide the EXACT official protein name as commonly used in scientific literature
- Choose the most concise, standard form of the protein name
- Avoid overly technical or lengthy variations (e.g., prefer "p53" over "Cellular tumor antigen p53")
- Return the primary, commonly used protein name
- Use standard nomenclature when available
- Examples: 
  - "TP53" → "p53"
  - "BRCA1" → "Breast cancer type 1 susceptibility protein"
  - "INS" → "Insulin"
- If the gene symbol is invalid or unknown, respond with "None"
- Use this exact JSON format:

{{
  "protein_name": "<protein_name_here>"
}}

Gene Symbol: {gene_symbol}"""

    response_text = query_together(prompt)
    result = extract_protein_data(response_text)

    # Log both the raw response and extracted result
    log_entry = {
        "gene_symbol": gene_symbol,
        "raw_response": response_text,
        "extracted_result": result
    }

    # Append to log file with your model name
    with open("gene_to_protein_api_responses_finetuned.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result.get("protein_name", "Error")

# Function to calculate match result (1 for match, 0 for no match)
def calculate_match(original_protein_name, new_protein_name):
    """
    Calculate match result between original and new protein names
    Returns 1 if they match, 0 if they don't match
    """
    if pd.isna(original_protein_name) or pd.isna(new_protein_name):
        return 0
    return int(str(original_protein_name).strip().lower() == str(new_protein_name).strip().lower())

# Main processing function
def main():
    print("Starting gene symbol to protein name mapping process with your fine-tuned model...")
    
    # Record start time for total processing
    total_start_time = time.time()

    # Create log file if it doesn't exist
    if not os.path.exists("gene_to_protein_api_responses_finetuned.jsonl"):
        with open("gene_to_protein_api_responses_finetuned.jsonl", "w") as f:
            pass

    # Test the API with a sample gene symbol
    print("Testing API connection...")
    test_result = get_protein_name_from_gene("TP53")
    print(f"Test result: {test_result}")

    if test_result == "Error" or test_result == "API_Error":
        print("API test failed. Please check your API key and connection.")
        return

    print("API test successful, proceeding with batch processing...")

    # Load the input data - using the protein names CSV
    input_file = r"C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv"  # Updated to use the uploaded file
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    df = pd.read_csv(input_file)
    print(f"Loaded {len(df)} rows from {input_file}")
    print(f"Columns in the dataset: {list(df.columns)}")

    # Add new result columns
    df["new_protein_name"] = None
    df["match_result"] = None
    df["processing_time"] = None

    # Process each row
    total_rows = len(df)
    for idx, row in df.iterrows():
        try:
            gene_symbol = row["GN"]  # Using the GN column from the CSV
            original_protein_name = row["protein_name"]  # Original protein name for comparison
            
            print(f"Processing {idx+1}/{total_rows}: {gene_symbol}")

            start_time = time.time()
            new_protein_name = get_protein_name_from_gene(gene_symbol)
            end_time = time.time()
            processing_time = end_time - start_time

            # Store the new protein name
            df.loc[idx, "new_protein_name"] = new_protein_name
            df.loc[idx, "processing_time"] = round(processing_time, 2)
            
            # Calculate match result (1 if match, 0 if no match)
            match_result = calculate_match(original_protein_name, new_protein_name)
            df.loc[idx, "match_result"] = match_result
            
            print(f"  Original: {original_protein_name}, New: {new_protein_name}, Match: {match_result}")

            # Save progress after every 10 rows or at the end
            if idx % 10 == 0 or idx == total_rows - 1:
                # Create progress dataframe excluding specified columns
                excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
                progress_columns = [col for col in df.columns if col not in excluded_columns]
                progress_df = df[progress_columns].copy()
                progress_df.to_csv("finetuned_gene_to_protein_results_progress1.csv", index=False)
                print(f"Progress saved. Processed {idx+1}/{total_rows} gene symbols.")

            # Add random delay between requests to avoid hammering the API
            if idx < total_rows - 1:
                delay = random.uniform(2, 4)  # Adjust delay as needed for your model
                print(f"Waiting {delay:.2f} seconds before next request...")
                time.sleep(delay)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            df.loc[idx, "new_protein_name"] = "Error"
            df.loc[idx, "match_result"] = 0
            df.loc[idx, "processing_time"] = 0

    # Create final results dataframe excluding specified columns
    excluded_columns = ['normalized_protein', 'normalized_gene_name', 'Match_GN']
    final_columns = [col for col in df.columns if col not in excluded_columns]
    final_df = df[final_columns].copy()
    
    # Save final results
    final_df.to_csv("finetuned_gene_to_protein_results_final1.csv", index=False)

    # Calculate match accuracy
    if "match_result" in df.columns:
        total_matches = df["match_result"].sum()
        total_processed = len(df)
        match_rate = (total_matches / total_processed) * 100
        
        print(f"\n=== FINAL RESULTS (Your Fine-tuned Model - Gene to Protein) ===")
        print(f"Total gene symbols processed: {total_processed}")
        print(f"Correct matches: {total_matches}")
        print(f"Match rate: {match_rate:.2f}%")
        
        # Count different result types
        none_results = (df["new_protein_name"] == "None").sum()
        error_results = (df["new_protein_name"] == "Error").sum()
        parse_error_results = (df["new_protein_name"] == "Parse_Error").sum()
        valid_results = total_processed - none_results - error_results - parse_error_results
        
        print(f"\nResults breakdown:")
        print(f"  - Valid protein names returned: {valid_results}")
        print(f"  - No match found (None): {none_results}")
        print(f"  - API errors: {error_results}")
        print(f"  - Parse errors: {parse_error_results}")
        
        # Calculate accuracy among valid results only
        if valid_results > 0:
            valid_matches = df[~df["new_protein_name"].isin(["None", "Error", "Parse_Error"])]["match_result"].sum()
            valid_accuracy = (valid_matches / valid_results) * 100
            print(f"  - Accuracy among valid results: {valid_accuracy:.2f}%")

        # Show some statistics about bins if available
        if 'bin' in df.columns:
            print(f"\nBreakdown by Bin:")
            bin_stats = df.groupby('bin').agg({
                'match_result': ['count', 'sum', 'mean']
            }).round(3)
            print(bin_stats)

    print(f"\nProcessing completed!")
    print(f"Final results saved to: finetuned_gene_to_protein_results_final5.csv")
    print(f"Progress file: finetuned_gene_to_protein_results_progress5.csv")
    print(f"Log file: gene_to_protein_api_responses_finetuned.jsonl")
    print(f"Columns excluded from output: normalized_protein, normalized_gene_name, Match_GN")

    # Calculate total time taken and overall accuracy - ADDED SECTION
    total_end_time = time.time()
    total_time_taken = total_end_time - total_start_time
    overall_accuracy = (total_matches / total_processed) * 100

    # Print the requested metrics
    print(f"\nTotal time taken: {total_time_taken:.2f} seconds")
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

if __name__ == "__main__":
    main()

Starting gene symbol to protein name mapping process with your fine-tuned model...
Testing API connection...
Sending request (attempt 1/5)...
Test result: p53
API test successful, proceeding with batch processing...
Loaded 3978 rows from C:\Users\sp5526s\OneDrive - Missouri State University\General - CLS Lab\Suswitha\Project Llama\protein_name_GN.csv
Columns in the dataset: ['ID', 'AC', 'GN', 'protein_name', 'GO_F Count', 'GO_P Count', 'GO_C Count', 'go_counts', 'bin', 'normalized_protein', 'normalized_gene_name', 'Match_GN', 'pmc_GN', 'PMC_protein_name']
Processing 1/3978: EXD2
Sending request (attempt 1/5)...
  Original: Exonuclease 3'-5' domain-containing protein 2 , New: Exodeoxyribonuclease II, Match: 0
Progress saved. Processed 1/3978 gene symbols.
Waiting 2.90 seconds before next request...
Processing 2/3978: EXOSC6
Sending request (attempt 1/5)...
  Original: Exosome complex component MTR3, New: Exosome complex component EXOSC6, Match: 0
Waiting 3.20 seconds before next request